In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

In [ ]:
me_updated = gpd.read_file("2012-2022_ME.csv")

In [ ]:
me_updated = me_updated.drop(
    columns=[
        "Unnamed: 0",
        "index__census",
        "OBJECTID_left",
        "LABEL",
        "ShapeSTArea",
        "ShapeSTLength",
        "OBJECTID_right",
        "Shape_Length",
        "Shape_Area",
        "FirstName",
        "MiddleName",
        "LastName",
    ]
)

In [ ]:
me_updated.to_csv("../data/latest_ME_data_clean_2012-2024.csv")

In [ ]:
# Extract the year
df = me_updated
df["DeathDate"] = pd.to_datetime(df["DeathDate"], errors="coerce")
df["Year"] = df["DeathDate"].dt.year
missing_dates = df["DeathDate"].isna().sum()
print(f"Number of records with missing DeathDate: {missing_dates}")

# Optionally, you can drop or impute these records
df = df.dropna(subset=["DeathDate"])
df["CT20"] = df["CT20"].astype(str)
df["ZIPCODE"] = df["ZIPCODE"].astype(str)

In [ ]:
years = list(range(2012, 2025))  # 2025 is not included

results = {
    "Year": [],
    "Total_People": [],
    "Excluded_People_CT20": [],
    "Excluded_People_ZIPCODE": [],
    "Percentage_Excluded_CT20": [],
    "Percentage_Excluded_ZIPCODE": [],
}
for year in years:
    # Filter data for the current year
    df_year = df[df["Year"] == year]
    total_people = len(df_year)

    if total_people == 0:
        # Skip years with no data
        continue

    # Group by CT20 and count
    ct20_counts = df_year.groupby("CT20").size().reset_index(name="Count")

    # Identify census tracts with fewer than 5 people
    low_count_ct20 = ct20_counts[ct20_counts["Count"] < 5]["CT20"]

    # Calculate number of people in these tracts
    num_excluded_ct20 = df_year[df_year["CT20"].isin(low_count_ct20)].shape[0]

    # Group by ZIPCODE and count
    zip_counts = df_year.groupby("ZIPCODE").size().reset_index(name="Count")

    # Identify zip codes with fewer than 5 people
    low_count_zip = zip_counts[zip_counts["Count"] < 5]["ZIPCODE"]

    # Calculate number of people in these zip codes
    num_excluded_zip = df_year[df_year["ZIPCODE"].isin(low_count_zip)].shape[0]

    # Calculate percentages
    perc_excluded_ct20 = (num_excluded_ct20 / total_people) * 100
    perc_excluded_zip = (num_excluded_zip / total_people) * 100

    # Store the results
    results["Year"].append(year)
    results["Total_People"].append(total_people)
    results["Excluded_People_CT20"].append(num_excluded_ct20)
    results["Excluded_People_ZIPCODE"].append(num_excluded_zip)
    results["Percentage_Excluded_CT20"].append(perc_excluded_ct20)
    results["Percentage_Excluded_ZIPCODE"].append(perc_excluded_zip)

In [ ]:
results_df = pd.DataFrame(results)

In [ ]:
results_df

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    results_df["Year"],
    results_df["Percentage_Excluded_CT20"],
    marker="o",
    label="Census Tracts",
)
plt.plot(
    results_df["Year"],
    results_df["Percentage_Excluded_ZIPCODE"],
    marker="o",
    label="ZIP Codes",
)

plt.title("Percentage of People Excluded by Year")
plt.xlabel("Year")
plt.ylabel("Percentage Excluded (%)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
total_people_all = len(df)

# Census tracts
ct20_counts_all = df.groupby("CT20").size().reset_index(name="Count")
low_count_ct20_all = ct20_counts_all[ct20_counts_all["Count"] < 5]["CT20"]
num_excluded_ct20_all = df[df["CT20"].isin(low_count_ct20_all)].shape[0]
perc_excluded_ct20_all = (num_excluded_ct20_all / total_people_all) * 100

# ZIP codes
zip_counts_all = df.groupby("ZIPCODE").size().reset_index(name="Count")
low_count_zip_all = zip_counts_all[zip_counts_all["Count"] < 5]["ZIPCODE"]
num_excluded_zip_all = df[df["ZIPCODE"].isin(low_count_zip_all)].shape[0]
perc_excluded_zip_all = (num_excluded_zip_all / total_people_all) * 100

print(f"Total People: {total_people_all}")
print(
    f"Excluded People (CT20): {num_excluded_ct20_all} ({perc_excluded_ct20_all:.2f}%)"
)
print(
    f"Excluded People (ZIPCODE): {num_excluded_zip_all} ({perc_excluded_zip_all:.2f}%)"
)